# AICS-106 — LSTM Sequence Model for Linux Auth Log Anomaly Detection

**Course:** AICS-106 Deep Learning for Threat Detection
**Author:** Janet Okewu-Ihezie
**Purpose:** Train an LSTM sequence model to classify Linux authentication sessions as Normal or Anomalous, using `auth_logs.csv`.

**Why a sequence model:** Unlike network flows (independent rows classified individually), auth log anomalies are defined by *patterns over time within a session* — e.g. repeated failed logins followed by success (credential stuffing), or a burst of `sudo_success` events (privilege escalation abuse). A single event in isolation looks harmless; the sequence and timing of events is what reveals the anomaly. This is exactly the kind of problem LSTMs are designed for.

**Handling class imbalance (per EDA findings):** Normal=88%, Anomalous=12%. We address this with:
1. Class-weighted loss function
2. Recall on the **Anomalous** class is treated as the most important metric — in a SOC context, missing a true anomaly (false negative) is far more costly than a false alarm (false positive)


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

sns.set_theme(style="whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Load & Build Sequences from Auth Logs

In [ ]:
df = pd.read_csv("auth_logs.csv", parse_dates=["timestamp"])
print("Shape:", df.shape)
df.head(10)

In [ ]:
# Encode event_type as integers (0 reserved for padding)
event_types = sorted(df["event_type"].unique())
event_to_idx = {evt: i + 1 for i, evt in enumerate(event_types)}  # start at 1, 0 = padding
print("Event vocabulary:", event_to_idx)

VOCAB_SIZE = len(event_to_idx) + 1  # +1 for padding token
MAX_SEQ_LEN = 12  # per EDA: session lengths range 2-12 events

df = df.sort_values(["session_id", "timestamp"]).reset_index(drop=True)
df["event_idx"] = df["event_type"].map(event_to_idx)

# Compute inter-event time gap (seconds) within each session -- a second input feature
df["time_gap"] = df.groupby("session_id")["timestamp"].diff().dt.total_seconds().fillna(0)
# Clip and log-scale time gaps to keep the feature well-behaved
df["time_gap_scaled"] = np.log1p(df["time_gap"].clip(lower=0)) / 10.0

df.head(10)

In [ ]:
# Build one sequence per session: list of (event_idx, time_gap_scaled) pairs, padded/truncated to MAX_SEQ_LEN
sessions = df.groupby("session_id")

X_event = []
X_time = []
y = []

for session_id, group in sessions:
    events = group["event_idx"].values[:MAX_SEQ_LEN]
    times = group["time_gap_scaled"].values[:MAX_SEQ_LEN]
    label = group["session_label"].iloc[0]

    seq_len = len(events)
    padded_events = np.zeros(MAX_SEQ_LEN, dtype=np.int64)
    padded_times = np.zeros(MAX_SEQ_LEN, dtype=np.float32)
    padded_events[:seq_len] = events
    padded_times[:seq_len] = times

    X_event.append(padded_events)
    X_time.append(padded_times)
    y.append(label)

X_event = np.array(X_event)
X_time = np.array(X_time)
y = np.array(y, dtype=np.int64)

print("X_event shape:", X_event.shape)
print("X_time shape:", X_time.shape)
print("y shape:", y.shape)
print("Label distribution:", dict(zip(*np.unique(y, return_counts=True))))

## 2. Train/Test Split & DataLoaders

In [ ]:
X_event_train, X_event_test, X_time_train, X_time_test, y_train, y_test = train_test_split(
    X_event, X_time, y, test_size=0.2, random_state=42, stratify=y
)

print("Train sessions:", len(y_train), "| Test sessions:", len(y_test))
print("Train label distribution:", dict(zip(*np.unique(y_train, return_counts=True))))

# Class weights (inverse frequency) for the imbalance identified in EDA (~88%/12%)
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * 2
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights [Normal, Anomalous]:", class_weights)

In [ ]:
def to_tensors(X_event, X_time, y):
    return (
        torch.tensor(X_event, dtype=torch.long),
        torch.tensor(X_time, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

Xe_train_t, Xt_train_t, y_train_t = to_tensors(X_event_train, X_time_train, y_train)
Xe_test_t, Xt_test_t, y_test_t = to_tensors(X_event_test, X_time_test, y_test)

train_dataset = TensorDataset(Xe_train_t, Xt_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = TensorDataset(Xe_test_t, Xt_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

## 3. Define the LSTM Sequence Model

Each event in a session is represented by two things: which event type it is (embedded into a dense vector) and how much time passed since the previous event (a scaled numeric feature). These are concatenated at each timestep and fed into an LSTM, which processes the sequence and produces a final hidden state summarizing the whole session. That summary is passed through a small classification head to predict Normal vs Anomalous.

In [ ]:
class AuthLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64, num_layers=2, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim + 1,  # +1 for the time_gap feature
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, event_seq, time_seq):
        embedded = self.embedding(event_seq)                       # (batch, seq_len, embed_dim)
        time_feat = time_seq.unsqueeze(-1)                         # (batch, seq_len, 1)
        lstm_input = torch.cat([embedded, time_feat], dim=-1)      # (batch, seq_len, embed_dim+1)

        lstm_out, (h_n, c_n) = self.lstm(lstm_input)
        # Concatenate final forward and backward hidden states
        final_hidden = torch.cat([h_n[-2], h_n[-1]], dim=-1)       # (batch, hidden_dim*2)

        return self.classifier(final_hidden)


model = AuthLSTM(vocab_size=VOCAB_SIZE, embed_dim=16, hidden_dim=64, num_layers=2, num_classes=2).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

## 4. Train the Model

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

NUM_EPOCHS = 20
train_losses, val_losses, val_f1_scores, val_anomaly_recalls = [], [], [], []

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xe, xt, yb in loader:
            xe, xt, yb = xe.to(device), xt.to(device), yb.to(device)
            out = model(xe, xt)
            loss = criterion(out, yb)
            total_loss += loss.item() * xe.size(0)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average="macro")
    anomaly_recall = recall_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1, anomaly_recall, all_preds, all_labels

print("Starting training...\n")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for xe, xt, yb in train_loader:
        xe, xt, yb = xe.to(device), xt.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xe, xt)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xe.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_f1, val_recall, _, _ = evaluate(test_loader)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1_scores.append(val_f1)
    val_anomaly_recalls.append(val_recall)

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Macro-F1: {val_f1:.4f} | Val Anomaly Recall: {val_recall:.4f}")

print("\nTraining complete.")

### 4.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(range(1, NUM_EPOCHS+1), train_losses, label="Train Loss", marker='o')
axes[0].plot(range(1, NUM_EPOCHS+1), val_losses, label="Validation Loss", marker='o')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training vs Validation Loss"); axes[0].legend()

axes[1].plot(range(1, NUM_EPOCHS+1), val_f1_scores, color="green", marker='o')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1-Score")
axes[1].set_title("Validation Macro-F1 Over Training")

axes[2].plot(range(1, NUM_EPOCHS+1), val_anomaly_recalls, color="crimson", marker='o')
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Anomaly Recall")
axes[2].set_title("Validation Anomaly-Class Recall Over Training")

plt.tight_layout()
plt.savefig("lstm_training_curves.png", dpi=150)
plt.show()

## 5. Final Evaluation — Per-Class Metrics

In [ ]:
final_loss, final_f1, final_recall, y_pred, y_true = evaluate(test_loader)

print("Final Test Macro-F1 Score:", round(final_f1, 4))
print("Final Test Anomaly Recall:", round(final_recall, 4))
print("\n" + "="*70)
print("CLASSIFICATION REPORT (Precision / Recall / F1 per class)")
print("="*70)
print(classification_report(y_true, y_pred, target_names=["Normal", "Anomalous"], digits=3))

**Why Anomaly Recall matters most here:** In a SOC context, a missed anomalous session (false negative) means a real credential-stuffing attack, off-hours privilege escalation, or sudo-abuse pattern goes undetected — a serious security failure. A false positive (flagging a normal session as anomalous) only costs an analyst a few minutes of review. This asymmetry is why recall on the Anomalous class, not overall accuracy, is the metric that matters most for this model.

### 5.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
labels = ["Normal", "Anomalous"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix — LSTM Auth Session Classifier")
plt.tight_layout()
plt.savefig("lstm_confusion_matrix.png", dpi=150)
plt.show()

## 6. Save the Trained Model

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "embed_dim": 16,
    "hidden_dim": 64,
    "num_layers": 2,
    "num_classes": 2,
    "max_seq_len": MAX_SEQ_LEN,
    "event_to_idx": event_to_idx,
}, "lstm_model.pt")

print("Model saved: lstm_model.pt")

## 7. Summary of Findings (for Capstone Report)

*(Fill in these bracketed values after running the notebook)*

1. **Final test macro-F1 score:** [X.XXX], **Anomaly-class recall:** [X.XXX] — recall on the Anomalous class is the primary metric, since missing a true anomaly is more costly than a false alarm in a SOC context.
2. **Sequence design:** Each session was represented as a sequence of up to 12 events (per EDA session-length findings), combining an embedded event-type vector with a scaled inter-event time gap at each timestep — allowing the model to learn both *what* events occurred and *how quickly* they occurred (critical for detecting rapid brute-force or sudo-abuse patterns).
3. **Class imbalance handling:** Class-weighted cross-entropy loss was used to address the ~88%/12% Normal/Anomalous imbalance identified in the EDA.
4. **Architecture justification:** A bidirectional 2-layer LSTM was used to capture patterns in both directions of the session timeline, which is particularly relevant for patterns like credential-stuffing (many failures *then* a success at the end of the session).
5. **Limitations:** [e.g. "Trained on synthetic sessions with three explicitly modeled anomaly patterns; real-world attacker behavior is more varied and would require retraining on live SOC data before production deployment."]
